# Predicting Kickstarter Project Success


In this notebook we build a **binary classification** model that predicts
whether a Kickstarter project will be **successful** or **failed**. We
will:

1. Load the preprocessed sample of the Kickstarter dataset.
2. Engineer features from raw columns (dates, videos, categories).
3. Set up two simple baselines that every model must beat.
4. Train gradient-boosted tree models with **XGBoost**, with and without a
   leaky feature.
5. Evaluate them with accuracy, ROC AUC, precision, recall, and F1, and
   discuss how those metrics are used in real-life analytics work.

:::{important} About the data and the results

This notebook uses a **small sample** of Kickstarter projects (about 2,700
finished campaigns). The sample has two problems that we keep on purpose,
because spotting them is a core analytics skill:

- **Data leakage.** One candidate feature, `backers_count`, is only known
  after a campaign ends.
- **A sampling artifact.** In this sample, many categories contain only
  successful projects or only failed projects, so the category alone almost
  gives away the answer.

Treat the headline numbers as a lesson in how a model can look good for the
wrong reasons, not as an estimate of how well success can be predicted on
all Kickstarter projects.

:::

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)

print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
print(f"xgboost version: {xgb.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")

pandas version: 3.1.1
numpy version: 2.5.4
xgboost version: 3.4.1
scikit-learn version: 1.9.1


:::{note} Reading Parquet files

pandas needs a Parquet engine such as `pyarrow` to read Parquet files. It is
included in the `requirements.txt` file at the root of the repository, which
installs everything this book uses:

```bash
python -m pip install -r requirements.txt
```

:::

## Load the Data

We work with a small, pre-cleaned subset of the Kickstarter dataset stored
as a Parquet file. Parquet is a columnar binary format that is much faster
to read than CSV and preserves data types (so date columns stay as
datetimes, etc.).

After loading, we use `df.head()` and `df.info()` to get a feel for the
columns, types, and non-null counts. This is always a good first move on
a new dataset.


In [2]:
df = pd.read_parquet(
    "https://media.githubusercontent.com/media/bdi593/datasets/main/kickstarter-projects/kickstarter-sample-preprocessed.parquet"
)
df.head()

,name,state,backers_count,usd_pledged,goal,percent_funded,launched_at,state_changed_at,country,currency,staff_pick,spotlight,category,category_parent,video,blurb,video_width,video_height
0,Stained Glass Mosaics,failed,4,13.000000,1500,0.866667,2016-07-19 22:35:04,2016-08-18 22:35:04,US,USD,False,False,Textiles,Art,NaN,Original works of art created with strong pass...,NaN,NaN
1,Personalized Boutique Balloons,failed,1,7.248741,1000,1.000000,2025-12-21 19:14:50,2026-01-31 23:50:00,CA,CAD,False,False,DIY,Crafts,NaN,You are the focus of luxury crafted arrangemen...,NaN,NaN
2,From Pop-Up to Permanent,failed,2,316.000000,4500,7.022222,2025-12-11 05:42:21,2026-01-15 05:42:21,US,USD,False,False,Candles,Crafts,NaN,Join us in building a studio home for Raven E'...,NaN,NaN
3,"43 Amp Arduino Motor shield, the Arc-Controller",failed,28,2565.000000,41500,6.180723,2014-08-14 06:02:45,2014-09-28 06:02:45,US,USD,False,False,DIY Electronics,Technology,"{""id"":426572,""status"":""successful"",""hls"":null,...",The Arc-Controller is here to give power to th...,640.0,360.0
4,The SnapPiCam | A Raspberry Pi Digital Camera.,failed,84,18503.697893,40000,27.340000,2014-08-01 09:41:01,2014-08-31 09:41:03,GB,GBP,True,False,DIY Electronics,Technology,NaN,Build your own Raspberry Pi powered Touchscree...,NaN,NaN


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2706 entries, 0 to 2705
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   name              2706 non-null   str           
 1   state             2706 non-null   str           
 2   backers_count     2706 non-null   int64         
 3   usd_pledged       2706 non-null   float64       
 4   goal              2706 non-null   int64         
 5   percent_funded    2706 non-null   float64       
 6   launched_at       2706 non-null   datetime64[us]
 7   state_changed_at  2706 non-null   datetime64[us]
 8   country           2706 non-null   str           
 9   currency          2706 non-null   str           
 10  staff_pick        2706 non-null   bool          
 11  spotlight         2706 non-null   bool          
 12  category          2706 non-null   str           
 13  category_parent   2654 non-null   str           
 14  video             1918 non-null   s

### Remove Duplicate Rows

A few projects appear twice in this file. If a duplicated project lands in
both the training set and the test set, the model is tested on a row it has
already seen. The [preprocessing notebook](./preprocess-data.ipynb) now
removes duplicate project ids, but the hosted copy of the file still contains
them, so we drop exact duplicate rows here before doing anything else.

In [4]:
print("Duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)
df.shape

Duplicate rows: 5


(2701, 18)

## Preprocess the Data for Binary Classification

Raw data is almost never in a shape a model can consume directly. In this
section we will:

1. Define a clean **target column** (success vs. failure).
2. Engineer a few **date features** from the launch timestamp.
3. Add a simple **video indicator**.
4. Clean up missing values in the **category** column.
5. Decide which **features** to include, and explicitly call out the ones
   that would cause data leakage.

Each of these steps mirrors what an analyst would do on a real project.

### Target Column

For binary classification we need a single column with two values: 1 for
the positive class and 0 for the negative class. The preprocessing notebook
kept only finished campaigns, so `state` has two values: `successful` and
`failed`.

We treat `successful` as the positive class (1) and `failed` as the
negative class (0). Before doing the conversion we look at
`value_counts()` to confirm the class balance. This quick sanity check
keeps us from accidentally training on, say, a column that is 99% one
class.

In [5]:
df["state"].value_counts()

state
successful    1671
failed        1030
Name: count, dtype: int64

In [6]:
df["state"].value_counts(normalize=True)

state
successful    0.61866
failed        0.38134
Name: proportion, dtype: float64

In [7]:
df["target"] = (df["state"] == "successful").astype(np.int64)

### Date Features

Models cannot consume raw timestamps directly: `2017-04-12 09:31:00` is
not a number the algorithm can compare. Instead, we decompose the launch
timestamp into a few interpretable components:

- **`launch_year`** captures slow-moving trends (the platform is more
  crowded in 2020 than in 2012).
- **`launch_month`** captures seasonality (e.g., a possible holiday-season
  boost).
- **`launch_dayofweek`** captures weekly patterns (campaigns launched on a
  Tuesday may behave differently from those launched on a Saturday).

This is a textbook example of **feature engineering**: turning raw data
into columns that an algorithm can use.

In [8]:
df["launch_year"] = df["launched_at"].dt.year
df["launch_month"] = df["launched_at"].dt.month
df["launch_dayofweek"] = df["launched_at"].dt.day_of_week

### Video Feature

The `video` column stores nested information about each project's pitch
video, but for this first model we only care whether a video exists at all.
We engineer a simple binary `has_video` indicator: a common pattern in real
analytics work where the *presence* of an asset (a photo, a phone number,
a verified email) is itself a strong signal even if we have not parsed the
content yet.


In [9]:
df["has_video"] = df["video"].notna().astype(np.int64)

### Category

Categorical columns like `category` and `category_parent` describe the type
of project (for example *Tabletop Games*, *Indie Rock*, *Documentary*).
They can carry real signal because backer communities form around
interests. In this sample, as we will see, `category` is suspiciously
strong.

The `"category_parent"` column contains the parent category of the project. The column contains missing (null) values, which we will fill with the string `"Unknown"`.


In [10]:
df["category_parent"] = df["category_parent"].fillna("Unknown")

### Feature Selection

Feature selection is the step where we decide which columns the model is
allowed to look at. There are three competing pressures:

1. **Predictive power**: we want features that actually carry signal about
   the outcome.
2. **Availability at prediction time**: a feature is only useful if we can
   compute it *before* we know the answer (otherwise we have data leakage).
3. **Stability and cost**: features that are noisy, expensive to compute,
   or change definitions over time tend to cause problems in production.

Our goal is to predict success **at launch**, so every feature should be
known on launch day. The list below follows that rule. We also build a
second list that adds `backers_count`, so that we can measure what a leaky
feature does to the results.

In [11]:
features = [
    "goal",
    "country",
    "currency",
    "staff_pick",
    "category",
    "category_parent",
    "has_video",
    "launch_year",
    "launch_month",
    "launch_dayofweek",
]

# Same list plus a leaky feature, kept separate on purpose (see the note below)
leaky_features = features + ["backers_count"]

:::{attention} Watch out for data leakage!

Data leakage occurs when the training data contains information that would not be available at prediction time. This leads to overly optimistic performance estimates during model evaluation, and poor performance on new data.

`backers_count` is the **final** number of backers. Every row in this dataset is a finished campaign, so the count was recorded after the outcome was already decided. A project cannot reach its goal without backers, so the final backer count is close to a restatement of the outcome. On launch day, when we would want to make the prediction, it is zero for every project.

We will train one model with `backers_count` and one without it, and compare them on the same test set.

:::

:::{note} Other excluded features

These columns are also excluded because their values are only final after the campaign ends:

- `usd_pledged`
- `percent_funded`
- `state_changed_at`
- `spotlight`: in this sample it is `True` for every successful project and `False` for every failed one, so it is a copy of the target (see the preprocessing notebook).

You could engineer features from some of these columns for a different question (for example, predicting success halfway through a campaign), but not for a prediction made at launch.

:::

### Select Features and Target Columns

By convention we use `X` for the feature matrix (the inputs the model will
look at) and `y` for the target column (what we are trying to predict).
Keeping this separation explicit makes it easy to swap in different feature
sets later when comparing model versions.


In [12]:
X = df[leaky_features].copy()
y = df["target"]

### Handle Categoricals

XGBoost can use categorical columns directly when we set
`enable_categorical=True` later, but we first need to convert the text
columns to the pandas `category` dtype. pandas 3 stores text with the
`str` dtype (you can see it in the `df.info()` output above), and XGBoost
does not accept `str` columns. The `category` dtype is also more memory
efficient and lets the model treat each level as its own group rather than
trying to interpret strings as numbers.

We list the text columns explicitly. This is clearer than selecting columns
by dtype, and it behaves the same in every pandas version.

In [13]:
categorical_cols = ["country", "currency", "category", "category_parent"]

for col in categorical_cols:
    X[col] = X[col].astype("category")

X.dtypes

goal                   int64
country             category
currency            category
staff_pick              bool
category            category
category_parent     category
has_video              int64
launch_year            int32
launch_month           int32
launch_dayofweek       int32
backers_count          int64
dtype: object

## Build a Model

With the features prepared, we are ready to train a binary classifier. The
workflow has three steps:

1. Split the data into training and test sets.
2. Fit the model on the training set.
3. Evaluate it on the held-out test set.

Keeping the evaluation set strictly separate from training is the single
most important habit in applied ML. It is how we avoid fooling ourselves
about how well the model really works.

### Split the Data into Training and Test Sets

Before training we set aside a test set the model will *never see* during
training. This gives us an estimate of how the model would perform on
projects it has not seen.

A few choices to highlight:

- **`test_size=0.2`**: keep 80% of the data for training and 20% for
  evaluation. This is a common starting point; with very small datasets you
  might use cross-validation instead.
- **`random_state=42`**: fixes the shuffling so the split is reproducible.
- **`stratify=y`**: ensures the train and test sets have the *same*
  proportion of successful vs. failed projects as the full dataset. This
  matters whenever the target is even slightly imbalanced; otherwise you
  can end up with a test set whose class mix does not match production.

In a real-life forecasting setting you would often go further and use a
**time-based split** (train on older projects, test on newer ones) to mimic
how the model will actually be used over time.

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### Baselines

Before training any model, check what simple rules achieve on the same test
set. A model is only useful if it beats them.

1. **Majority class**: predict the most common outcome in the training set
   (success) for every project.
2. **Category majority**: for each category, predict the outcome that was
   more common for that category in the training set. Projects in a category
   that never appears in the training set get the overall majority class.

We collect every result in a list called `results` so that we can compare
all models in one table at the end.

In [15]:
results = []


def record(name, y_true, y_pred, y_prob=None):
    """Store test-set metrics for one model or rule."""
    results.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_true, y_pred),
            "f1 (successful)": f1_score(y_true, y_pred),
            "roc_auc": roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan,
        }
    )


# Baseline 1: always predict the majority class of the training set
majority_class = int(y_train.mode()[0])
record("Majority class", y_test, np.full(len(y_test), majority_class))

# Baseline 2: predict each category's majority outcome in the training set
train_category = X_train["category"].astype(str)
test_category = X_test["category"].astype(str)
category_majority = y_train.groupby(train_category).mean().ge(0.5).astype(int)
y_pred_category = (
    test_category.map(category_majority).fillna(majority_class).astype(int)
)
record("Category majority", y_test, y_pred_category)

pd.DataFrame(results).round(3)

,model,accuracy,f1 (successful),roc_auc
0,Majority class,0.619,0.765,NaN
1,Category majority,0.915,0.931,NaN


Always predicting success is right about 62% of the time, because about 62%
of the projects in this sample succeeded. That is the floor.

The category rule is far stronger than you might expect: it is right about
nine times out of ten, using nothing but the project's category. ROC AUC is
blank for both rules because they output hard 0/1 labels, not
probabilities. Keep the category result in mind; we come back to it in the
category analysis below.

### Train the XGBoost Models

We use **XGBoost**, a gradient-boosted decision tree library that is a very
common first choice for tabular classification problems in industry. It
tends to work well out of the box, handles mixed numeric and categorical
features (when `enable_categorical=True`), handles missing values, and
trains quickly even on large datasets.

A few defaults worth knowing about:

- `random_state=42` makes the run reproducible.
- `enable_categorical=True` lets us pass the pandas `category` columns we
  created above directly into the model, with no manual one-hot encoding.
- We are not tuning any hyperparameters in this first pass. In a real
  project, we would later do cross-validated hyperparameter search (for
  example over `max_depth`, `learning_rate`, `n_estimators`, and
  `min_child_weight`) to squeeze out more performance.

We train two models with identical settings on the same training rows. The
only difference is whether `backers_count` is included.

In [16]:
def train_xgb(columns):
    """Fit an XGBoost classifier on the given feature columns."""
    clf = xgb.XGBClassifier(enable_categorical=True, random_state=42)
    clf.fit(X_train[columns], y_train)
    return clf


model_leaky = train_xgb(leaky_features)
model = train_xgb(features)

## Evaluate the Models

Training accuracy is not a trustworthy measure of how well a model will do
on new projects, because a model can memorize its training set. To get an
honest estimate of generalization performance we score the models on the
held-out test set we set aside earlier.

In real-life analytics work, evaluation is rarely a single number. We
typically:

1. Look at multiple metrics so we understand *which kind* of mistake the
   model is making (false positives vs. false negatives).
2. Compare against simple baselines to confirm the model is actually adding
   value. We set these up above.
3. Slice performance by important sub-populations (category, country,
   year) to make sure the model is not great on average but poor for an
   important segment. We do a version of this for category below.

### Model Performance

We use each trained model to produce two outputs for every test row:

- `y_pred`: a hard 0/1 prediction (predict success when the probability is
  at least 0.5).
- `y_prob`: the predicted probability of the positive class (success).

We need the probabilities to compute ROC AUC.

In [17]:
for name, clf, columns in [
    ("XGBoost with backers_count (leaky)", model_leaky, leaky_features),
    ("XGBoost without backers_count", model, features),
]:
    y_prob = clf.predict_proba(X_test[columns])[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    record(name, y_test, y_pred, y_prob)

pd.DataFrame(results).round(3)

,model,accuracy,f1 (successful),roc_auc
0,Majority class,0.619,0.765,NaN
1,Category majority,0.915,0.931,NaN
2,XGBoost with backers_count (leaky),0.967,0.973,0.996
3,XGBoost without backers_count,0.943,0.954,0.984


The model with `backers_count` scores highest, with an accuracy of about
0.97 and a near-perfect ROC AUC. That is the leak at work: the model is
partly reading the answer. Without `backers_count` the accuracy drops by a
few points, to about 0.94.

The drop is smaller than you might expect, because the category alone
already carries most of the answer in this sample (compare the category
majority row). From here on we treat the model **without** `backers_count`
as our model, because it is the only one we could use on launch day.

Here is its full classification report.

In [18]:
y_pred = model.predict(X_test[features])
print(classification_report(y_test, y_pred, target_names=["failed", "successful"]))

              precision    recall  f1-score   support

      failed       0.92      0.93      0.92       206
  successful       0.96      0.95      0.95       335

    accuracy                           0.94       541
   macro avg       0.94      0.94      0.94       541
weighted avg       0.94      0.94      0.94       541



#### Interpretation of Model Performance Metrics

For the model without `backers_count`, the report shows:

- Precision and recall are above 0.9 for both classes, so the model
  rarely misclassifies either class on this test set.
- The model does slightly better on successful projects than on failed
  ones (compare the F1-scores of the two classes).
- The macro and weighted averages are close to each other because the two
  classes are not far from balanced (about 62% vs. 38%).

These numbers look excellent, but remember that the category majority rule
gets most of the way there by itself. The category analysis below explains
why.

**What the Metrics Mean**

- **Support**: Number of actual observations in each class (how many true
  failed vs. successful projects in the test set).
- **Precision**: Of the projects predicted as a given class, the proportion
  that were actually correct. *Precision answers: "When the model says yes,
  how often is it right?"*
- **Recall**: Of all true projects in a class, the proportion the model
  successfully identified. *Recall answers: "Of the things we cared about,
  how many did we catch?"*
- **F1-Score**: The harmonic mean of precision and recall. It balances false
  positives and false negatives. Useful when classes are imbalanced or when
  both error types are costly.
- **Accuracy**: The fraction of all predictions that were correct. Easy to
  explain, but can be misleading when one class dominates (e.g., a model
  that always predicts "failed" on a 95%-failure dataset would still hit
  95% accuracy).
- **ROC AUC**: Measures how well the model *ranks* successes above failures
  across every possible probability cutoff. A value of 0.5 is random and 1.0
  is perfect. AUC is threshold-independent, so it is useful when you have
  not yet decided what probability cutoff to use in production.

**How these metrics show up in real-life analytics**

The "right" metric depends on the business cost of each kind of mistake.

- *Fraud detection / churn alerts.* The cost of missing a true positive
  (letting a fraudster through, losing a customer silently) is typically
  much higher than the cost of a false alarm. Analytics teams optimize for
  **recall** on the positive class, often accepting lower precision and
  routing flagged cases to a human reviewer.
- *Marketing targeting / sales lead scoring.* Outreach has a per-contact
  cost (emails, sales rep time), so we want most of the people we *do*
  contact to actually convert. That means we optimize for **precision** on
  the positive class.
- *Crowdfunding success prediction (this case).* If we built a "should I
  back this project?" recommender, false positives (recommending a doomed
  project) have a real reputational cost, so we would care about precision.
  If we built a tool that *flagged risky projects for the Kickstarter trust
  & safety team*, false negatives would matter more, so recall would win.
- *Regulated or imbalanced settings.* In medical screening, credit
  decisions, or anti-money-laundering, teams almost always report
  precision, recall, F1, and AUC together, and frequently look at the full
  precision-recall curve at multiple thresholds, because a single number
  cannot capture the trade-off.

A good rule of thumb: **decide which mistake is more expensive *before* you
look at the metrics, then pick the threshold and metric that match that
trade-off.**

### Feature Importance


In [19]:
df_feature_importances = (
    pd.DataFrame(
        {
            "with backers_count": pd.Series(
                model_leaky.feature_importances_, index=leaky_features
            ),
            "without backers_count": pd.Series(
                model.feature_importances_, index=features
            ),
        }
    )
    .sort_values("without backers_count", ascending=False)
    .round(3)
)

df_feature_importances

,with backers_count,without backers_count
category,0.532,0.412
staff_pick,0.016,0.198
category_parent,0.043,0.102
launch_year,0.043,0.074
country,0.016,0.054
has_video,0.001,0.045
goal,0.072,0.034
launch_month,0.010,0.029
currency,0.012,0.027
launch_dayofweek,0.009,0.025


#### Interpretation of Feature Importance

Feature importance tells us how much each input contributed to the model's
splits. It is one of the most common ways analysts translate a "black box"
model into something a business stakeholder can act on.

The table shows the importance of each feature in both models. Each column
sums to 1. Reading it:

- **`category` dominates both models.** That would be a reasonable finding
  in a well-drawn sample. Here it is a warning sign, as the next section
  shows.
- **`backers_count` takes a large share of the leaky model's importance.**
  A single feature with a suspiciously high importance is one of the most
  reliable warning signs of leakage.
- **`staff_pick` gains a lot of importance once `backers_count` is
  removed.** When a strong feature is dropped, the model shifts weight to the
  features that are correlated with it. Staff picks succeed far more often
  than other projects in this sample.
- The date and video features carry little weight in either model.

**How analysts use this in practice**

- *Validate intuition.* If the most important features match what subject
  matter experts expect, that is a sanity check. If they do not, it is a
  signal to investigate (often a leakage bug, like `backers_count` here).
- *Identify leakage.* A feature that would not be known at prediction time
  should never appear near the top of this table.
- *Prioritize data collection.* Features that show up at the top are worth
  investing in: better cleaning, more frequent refresh, or richer encodings.
- *Trim noise.* Features at the bottom can often be dropped to simplify the
  model and reduce maintenance cost without losing accuracy.

`feature_importances_` in XGBoost's scikit-learn interface uses **gain** by
default for tree models: the average improvement in the loss from the
splits that use each feature, scaled so the values sum to 1. You can ask for
other definitions, such as `importance_type="weight"` (how often a feature
is used to split). Importance does *not* tell us the direction of the
effect (does a high goal increase or decrease success?). For that we would
normally use SHAP values or partial dependence plots.

### Category Analysis


In [20]:
df.groupby(["category", "state"], as_index=False).size()

,category,state,size
0,3D Printing,successful,12
1,Academic,failed,18
2,Accessories,successful,22
3,Action,failed,47
4,Action,successful,3
...,...,...,...
178,World Music,failed,1
179,World Music,successful,33
180,Young Adult,failed,54
181,Zines,failed,32


In [21]:
category_summary = df.groupby("category").agg(
    total_projects=("state", "count"),
    successes=("target", "sum"),
    success_rate=("target", "mean"),
)

category_summary.head(10)

,total_projects,successes,success_rate
category,,,
3D Printing,12,12,1.00
Academic,18,0,0.00
Accessories,22,22,1.00
Action,50,3,0.06
Animals,1,1,1.00
Animation,25,7,0.28
Anthologies,67,67,1.00
Apparel,36,36,1.00
Apps,10,0,0.00


In [22]:
single_outcome = category_summary["success_rate"].isin([0, 1])

print(f"Categories: {len(category_summary)}")
print(f"Categories with only one outcome: {single_outcome.sum()}")
print(
    "Projects in those categories:",
    category_summary.loc[single_outcome, "total_projects"].sum(),
    "of",
    len(df),
)

Categories: 124
Categories with only one outcome: 65
Projects in those categories: 1026 of 2701


More than half of the categories contain only successful projects or only
failed projects. "Anthologies" and "Apparel" are 100% successful, while
"Academic" and "Apps" are 0% successful. Real categories rarely look like
this: in a random sample of all Kickstarter projects you would expect a mix
of outcomes in almost every category. A pattern this extreme suggests the
sample was not drawn at random. The [Web Robots
page](https://webrobots.io/kickstarter-datasets/) that publishes the raw
data notes that Kickstarter limits how many projects a visitor can view in a
single category, and that its scraper walks through the site category by
category. A cap per category, combined with the order in which each
category lists its projects, is one way a sample can end up skewed like
this.

Whatever the cause, the effect on modeling is the same. With a random
train/test split, projects from the same single-outcome category land on
both sides, and the model can memorize "category X always succeeds". That
skill would not transfer to new projects.

To see how much the model depends on this, train it once more without
`category` and `category_parent`.

In [23]:
features_no_category = [
    f for f in features if f not in ["category", "category_parent"]
]
model_no_category = train_xgb(features_no_category)

y_prob = model_no_category.predict_proba(X_test[features_no_category])[:, 1]
record(
    "XGBoost without backers_count or category",
    y_test,
    (y_prob >= 0.5).astype(int),
    y_prob,
)

df_results = pd.DataFrame(results).set_index("model").round(3)
df_results

,accuracy,f1 (successful),roc_auc
model,,,
Majority class,0.619,0.765,NaN
Category majority,0.915,0.931,NaN
XGBoost with backers_count (leaky),0.967,0.973,0.996
XGBoost without backers_count,0.943,0.954,0.984
XGBoost without backers_count or category,0.706,0.767,0.784


:::{attention} Why does this model look so good?

The accuracy and ROC AUC of our models are unusually high for a real-world
prediction problem. The comparison table above shows why.

1. **`backers_count` leaks the outcome.** It is the final number of
   backers, recorded after the campaign ended. Including it lifts accuracy
   by a few points and pushes ROC AUC close to 1.0. A model built this way
   cannot be used on launch day.
2. **The sample has a category artifact.** More than half of the categories
   contain only one outcome. The category majority rule alone is right about
   nine times out of ten, close to the XGBoost model. When we remove the two
   category columns, accuracy falls to about 0.7. That is a more realistic
   picture of what launch-day information such as goal, country, staff
   pick, and video can do.
3. **The split is random, not time-based.** Launch dates in this sample
   span 2009 to 2026, so the model trains on projects launched *after* some
   of the test projects. A time-based split would be a fairer test of how
   the model would be used.

The lesson is not that XGBoost is bad. It is that a model is only as honest
as its features and its sample. Always compare against simple baselines,
ask whether each feature is known at prediction time, and check whether the
sample looks like the population you care about.

:::

## Summary

- We removed five duplicate rows, then split about 2,700 finished
  campaigns into training (80%) and test (20%) sets, stratified by outcome.
- A majority-class rule scores about 62% accuracy and a category majority
  rule about 91%, on the same test set as the models.
- XGBoost with the leaky `backers_count` feature reaches about 97% accuracy
  and a ROC AUC near 1.0. Without it, accuracy is about 94%. Only the model
  without `backers_count` could be used on launch day.
- Most of the remaining accuracy comes from the category artifact in this
  sample. Without the category columns, accuracy drops to about 71%.
- `feature_importances_` uses gain by default. In both models `category`
  dominates, which here is a symptom of the sampling artifact, not a
  business insight.